# Validador de estado atual — monorepo-ai-llm (NestJS CLI)

Este notebook **não altera arquivos** (somente leitura) e serve para você entender **o que já foi aplicado** no repo.

Ele valida e reporta:
- Quais apps existem em `apps/`
- Conteúdo do `nest-cli.json` (projetos registrados)
- Se cada app tem `HealthController` e está registrado no `AppModule`
- Se cada app tem porta default (fallback) coerente no `main.ts`
- Se existe `.env.containers.example` e como o compose está estruturado (se existir)
- Se existem arquivos de governança (`spec/`, `.github/instructions/`, `llms.txt`, etc.)

## Como usar
1. Salve este arquivo como `notebooks/02_validate_current_config_state.ipynb`
2. Abra no Jupyter / VS Code Notebook
3. Execute as células em ordem


In [3]:
from __future__ import annotations

from pathlib import Path
import json
import re
from dataclasses import dataclass
from typing import Any

def find_repo_root(start: Path) -> Path:
    candidates = [start, *start.parents]
    for base in candidates:
        if (base / 'package.json').exists() and (base / 'apps').exists():
            return base
        if (base / 'nest-cli.json').exists():
            return base
    return start

ROOT = find_repo_root(Path.cwd().resolve())


In [4]:

@dataclass(frozen=True)
class AppExpectation:
    name: str
    expected_port: int
    required: bool = True

# Ajuste aqui se você quiser incluir/excluir apps
EXPECTED_APPS: list[AppExpectation] = [
    AppExpectation('users-api', 3001),
    AppExpectation('llm-ops-api', 3002),
    AppExpectation('sharepoint-api', 3003),
    AppExpectation('sync-api', 3004),
    AppExpectation('monorepo-ai-llm', 3000, required=False),
]

def read_text(p: Path) -> str:
    return p.read_text(encoding='utf-8')

def safe_read_text(p: Path) -> str | None:
    if not p.exists():
        return None
    return read_text(p)

def safe_read_json(p: Path) -> Any | None:
    if not p.exists():
        return None
    return json.loads(read_text(p))

print('Repo root:', ROOT)

Repo root: /home/daniel/monorepo-ai-llm


## 1) Snapshot: árvore e arquivos-chave

Mostra rapidamente o que existe na raiz.

In [11]:
key_paths = [
    'package.json',
    'nest-cli.json',
    'tsconfig.json',
    'docker-compose.yml',
    '.env.containers',
    '.env.containers.example',
    'llms.txt',
    'spec',
    'docs',
    '.github/instructions',
    '.github/prompts',
    '.github/skills',
    'apps',
    'packages',
    'tools',
]

for rp in key_paths:
    p = ROOT / rp
    print(('✅' if p.exists() else '❌'), rp)

✅ package.json
✅ nest-cli.json
✅ tsconfig.json
❌ docker-compose.yml
❌ .env.containers
❌ .env.containers.example
❌ llms.txt
✅ spec
✅ docs
✅ .github/instructions
✅ .github/prompts
✅ .github/skills
✅ apps
✅ packages
✅ tools


## 2) Apps detectados em `apps/`

In [5]:
apps_dir = ROOT / 'apps'
apps = []
if apps_dir.exists():
    apps = sorted([p.name for p in apps_dir.iterdir() if p.is_dir()])

print('Apps encontrados:', apps)

missing_required = []
for exp in EXPECTED_APPS:
    if exp.required and exp.name not in apps:
        missing_required.append(exp.name)

if missing_required:
    print('❌ Faltando apps obrigatórios:', missing_required)
else:
    print('✅ Apps obrigatórios presentes')

Apps encontrados: ['llm-ops-api', 'monorepo-ai-llm', 'sharepoint-api', 'svcia', 'sync-api', 'users-api']
✅ Apps obrigatórios presentes


## 3) `nest-cli.json` (projetos registrados)

Mostra se cada app está registrado como projeto do Nest monorepo.

In [6]:
nest_cli = safe_read_json(ROOT / 'nest-cli.json')
if not nest_cli:
    print('❌ nest-cli.json não encontrado')
else:
    print('monorepo:', nest_cli.get('monorepo'))
    print('root:', nest_cli.get('root'))
    print('sourceRoot:', nest_cli.get('sourceRoot'))
    projects = nest_cli.get('projects', {})
    print('\nProjetos registrados:')
    for k in sorted(projects.keys()):
        print(' -', k)

    print('\nValidação dos apps esperados:')
    for exp in EXPECTED_APPS:
        status = '✅' if exp.name in projects else ('⚠️' if not exp.required else '❌')
        print(status, exp.name)

monorepo: True
root: apps/monorepo-ai-llm
sourceRoot: apps/monorepo-ai-llm/src

Projetos registrados:
 - llm-ops-api
 - monorepo-ai-llm
 - sharepoint-api
 - sync-api
 - users-api

Validação dos apps esperados:
✅ users-api
✅ llm-ops-api
✅ sharepoint-api
✅ sync-api
✅ monorepo-ai-llm


## 4) Validação por app: `HealthController`, `AppModule`, `main.ts`

Checa:
- existe `health.controller.ts`
- `app.module.ts` importa e registra `HealthController`
- `main.ts` usa `process.env.PORT` e tem fallback com porta esperada (heurístico)


In [7]:
def check_app(app: AppExpectation) -> dict:
    app_src = ROOT / 'apps' / app.name / 'src'
    result: dict[str, object] = {
        'app': app.name,
        'exists': app_src.exists(),
        'expected_port': app.expected_port,
        'health_controller_exists': False,
        'app_module_exists': False,
        'app_module_imports_health': False,
        'app_module_registers_health': False,
        'app_module_exports_class': False,
        'main_exists': False,
        'main_uses_env_port': False,
        'main_has_expected_fallback': False,
        'notes': [],
    }
    if not app_src.exists():
        return result

    hc = app_src / 'health.controller.ts'
    result['health_controller_exists'] = hc.exists()

    am = app_src / 'app.module.ts'
    result['app_module_exists'] = am.exists()
    if am.exists():
        s = read_text(am)
        result['app_module_imports_health'] = "./health.controller" in s
        result['app_module_registers_health'] = "HealthController" in s and "controllers" in s
        result['app_module_exports_class'] = re.search(r"export\s+class\s+AppModule", s) is not None

    mt = app_src / 'main.ts'
    result['main_exists'] = mt.exists()
    if mt.exists():
        s = read_text(mt)
        result['main_uses_env_port'] = "process.env.PORT" in s
        # heurística simples: procura o número da porta no arquivo
        result['main_has_expected_fallback'] = str(app.expected_port) in s

    # notas
    if not result['health_controller_exists']:
        result['notes'].append('HealthController ausente')
    if result['app_module_exists'] and not result['app_module_registers_health']:
        result['notes'].append('AppModule não registra HealthController')
    if result['main_exists'] and not result['main_uses_env_port']:
        result['notes'].append('main.ts não usa process.env.PORT')
    if result['main_exists'] and not result['main_has_expected_fallback']:
        result['notes'].append('main.ts pode não ter fallback de porta esperado (heurístico)')

    return result

results = [check_app(a) for a in EXPECTED_APPS]
for r in results:
    if not r['exists']:
        icon = '⚠️' if next(x for x in EXPECTED_APPS if x.name == r['app']).required is False else '❌'
        print(f"{icon} {r['app']} não encontrado")
        continue
    ok = (
        r['health_controller_exists']
        and r['app_module_exists']
        and r['app_module_imports_health']
        and r['app_module_registers_health']
        and r['app_module_exports_class']
        and r['main_exists']
        and r['main_uses_env_port']
    )
    print(('✅' if ok else '❌'), r['app'], '| notes:', '; '.join(r['notes']) if r['notes'] else '-')

✅ users-api | notes: -
✅ llm-ops-api | notes: -
✅ sharepoint-api | notes: -
✅ sync-api | notes: -
❌ monorepo-ai-llm | notes: AppModule não registra HealthController


## 5) Governança: specs, instruções, prompts, skills

Checa arquivos-base usando caminhos canônicos e variações de nome já presentes no repositório.

In [8]:
governance_checks = {
    'spec de boundaries': [
        'spec/spec-microservice-packaging-and-boundaries.md',
        'spec/spec_spec-microservice-packaging-and-boundaries.md',
    ],
    'instructions (packaging)': [
        '.github/instructions/microservice-packaging.instructions.md',
    ],
    'instructions (memory)': [
        '.github/instructions/microservice-packaging-memory.instructions.md',
    ],
    'prompt review': [
        '.github/prompts/microservice-guardian-review.prompt.md',
    ],
    'skill guardian': [
        '.github/skills/microservice-guardian/skill.md',
        '.github/skills/microservice-guardian/SKILL.md',
    ],
    'blueprint arquitetura': [
        'docs/architecture/project-structure-blueprint.md',
        'spec/docs_architecture_project-structure-blueprint.md',
    ],
    'plano naming': [
        'docs/architecture/naming-conventions-plan.md',
        'spec/docs_naming-conventions-plan.md',
    ],
    'runbook low-resource': [
        'docs/runbooks/low-resource-machine.md',
    ],
    'llms': [
        'llms.txt',
    ],
    'env containers example': [
        '.env.containers.example',
    ],
}

for label, candidates in governance_checks.items():
    found = next((rel for rel in candidates if (ROOT / rel).exists()), None)
    if found:
        print('✅', f'{label}:', found)
    else:
        print('❌', f'{label}:', 'não encontrado')

✅ spec de boundaries: spec/spec_spec-microservice-packaging-and-boundaries.md
❌ instructions (packaging): não encontrado
❌ instructions (memory): não encontrado
❌ prompt review: não encontrado
❌ skill guardian: não encontrado
✅ blueprint arquitetura: spec/docs_architecture_project-structure-blueprint.md
✅ plano naming: spec/docs_naming-conventions-plan.md
❌ runbook low-resource: não encontrado
❌ llms: não encontrado
❌ env containers example: não encontrado


## 6) Compose/Docker: leitura rápida (se existir)

Mostra trechos úteis para entender se:
- há perfis `persistent`/`demand`
- há limites de recursos (`mem_limit`, `cpus`)


In [16]:
compose_path = ROOT / 'docker-compose.yml'
if not compose_path.exists():
    print('⚠️ docker-compose.yml não encontrado')
else:
    s = read_text(compose_path)
    print('docker-compose.yml encontrado. Heurísticas:')
    print(' - contém "profiles":', 'profiles' in s)
    print(' - contém "persistent":', 'persistent' in s)
    print(' - contém "demand":', 'demand' in s)
    print(' - contém "mem_limit":', 'mem_limit' in s)
    print(' - contém "cpus":', 'cpus' in s)

    # imprime primeiros 80 linhas
    print('\n--- head (80) ---')
    print('\n'.join(s.splitlines()[:80]))

⚠️ docker-compose.yml não encontrado


## 7) Próximos passos sugeridos (baseado no report)

Se algum app falhar na validação:
- criar `health.controller.ts`
- registrar no `app.module.ts`
- ajustar `main.ts` com porta e `process.env.PORT`

Se governança estiver faltando:
- criar `spec/` e `.github/instructions/` etc.

Se compose não existir:
- gerar `docker-compose.yml` com perfis + limites (evitar derrubar desktop)
